In [1]:
########################################################################
# 1. 依赖 & 配置
########################################################################
import os, random, time
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from textblob import TextBlob

# Reproducibility & Device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {DEVICE}")

########################################################################
# 2. 通用函数定义
########################################################################

def parse_impr(s):
    return [(p.split('-')[0], int(p.split('-')[1])) for p in str(s).split()]

def parse_hist(h):
    return str(h).split()

def split_by_impression(df, test_ratio=0.2):
    df_sorted = df.sort_values("Time")
    cut = int(len(df_sorted) * (1 - test_ratio))
    return df_sorted.iloc[:cut], df_sorted.iloc[cut:]

def generate_balanced_samples(df, uid2idx, news2idx, neg_ratio=4):
    pos_u, pos_n, neg_u, neg_n = [], [], [], []
    all_ids = list(news2idx.values())
    for _, row in df.iterrows():
        uidx = uid2idx[row.User_ID]
        hist_ids = [news2idx[n] for n in row.History_parsed if n in news2idx]
        for hid in hist_ids:
            pos_u.append(uidx); pos_n.append(hid)
        im_pos, im_neg = [], []
        for nid, lbl in row.Impressions_parsed:
            if nid in news2idx:
                (im_pos if lbl==1 else im_neg).append(news2idx[nid])
        for nid in im_pos:
            pos_u.append(uidx); pos_n.append(nid)
        pool = [n for n in im_neg if n not in hist_ids]
        for _ in range(len(im_pos)*neg_ratio):
            if pool:
                choice = random.choice(pool)
            else:
                candidates = set(all_ids) - set(im_pos) - set(hist_ids)
                if not candidates: break
                choice = random.choice(list(candidates))
            neg_u.append(uidx); neg_n.append(choice)
    return pos_u, pos_n, neg_u, neg_n

########################################################################
# 3. 数据加载 & 预处理
########################################################################
# 3.1 读取文件 & 抽样
news = pd.read_csv(
    '/kaggle/input/mind-news-dataset/MINDsmall_train/news.tsv', sep='\t',
    names=["News_ID","Category","SubCategory","Title","Abstract",
           "URL","Title_Entities","Abstract_Entities"]
)
beh = pd.read_csv(
    '/kaggle/input/mind-news-dataset/MINDsmall_train/behaviors.tsv', sep='\t',
    names=["Impression_ID","User_ID","Time","History","Impressions"]
)
beh = beh.sample(frac=0.20, random_state=SEED).reset_index(drop=True)
print(f"Sampled behaviors: {len(beh):,} rows")

# 3.2 解析列
beh["Impressions_parsed"] = beh["Impressions"].apply(parse_impr)
beh["History_parsed"] = beh["History"].apply(parse_hist)

# 3.3 过滤 News
used_nids = set(nid for lst in beh["Impressions_parsed"] for nid,_ in lst)
used_nids |= set(n for lst in beh["History_parsed"] for n in lst)
news = news[news["News_ID"].isin(used_nids)].reset_index(drop=True)
print(f"Retained news: {len(news):,} articles")

# 3.4 Train/Val 切分
df_train, df_val = split_by_impression(beh, test_ratio=0.2)

########################################################################
# 4. 特征工程 (情感 + 类别 one-hot + TF-IDF + SBERT)
########################################################################
# 4.1 情感分数
news["sent"] = news["Title"].apply(lambda t: TextBlob(str(t)).sentiment.polarity)

# 4.2 类别 one-hot
cats = news.Category.unique().tolist()
cat2idx = {c:i for i,c in enumerate(cats)}
news_cat = np.zeros((len(news), len(cats)))
for i,c in enumerate(news.Category):
    news_cat[i, cat2idx[c]] = 1

# 4.3 TF-IDF (仅训练集)
train_nids = set(n for lst in df_train.Impressions_parsed for n,_ in lst) | set(n for lst in df_train.History_parsed     for n   in lst)
train_mask = news.News_ID.isin(train_nids)
val_mask   = ~train_mask

tfv = TfidfVectorizer(max_features=100)
tfidf_train = tfv.fit_transform(
    news.loc[train_mask, "Title"].fillna("")
).toarray()
tfidf_val = tfv.transform(
    news.loc[val_mask, "Title"].fillna("")
).toarray()
# 合并
tfidf = np.zeros((len(news), tfidf_train.shape[1]))
tfidf[train_mask] = tfidf_train
tfidf[val_mask]   = tfidf_val

# 4.4 SBERT 嵌入 & 标准化
sbert = SentenceTransformer("paraphrase-MiniLM-L6-v2")
txt_train = (
    news.loc[train_mask, ["Title","Abstract"]]
        .fillna("")
        .apply(lambda x: " ".join(x), axis=1)
        .tolist()
)
bt = sbert.encode(txt_train, batch_size=128)
scaler = StandardScaler().fit(bt)
bert_train = scaler.transform(bt)
bert_val   = scaler.transform(
    sbert.encode(
        news.loc[val_mask,["Title","Abstract"]]
            .fillna("")
            .apply(lambda x: " ".join(x), axis=1)
            .tolist(),
        batch_size=128
    )
)
# 合并
bert = np.zeros((len(news), bert_train.shape[1]))
bert[train_mask] = bert_train
bert[val_mask]   = bert_val

# 4.5 拼接特征
news_x = torch.tensor(
    np.hstack([news.sent.values.reshape(-1,1), news_cat, tfidf, bert]),
    dtype=torch.float
)
print(f"news_x shape: {news_x.shape}")

########################################################################
# 5. 索引映射
########################################################################
news2idx = {nid:i for i,nid in enumerate(news.News_ID)}
uid2idx  = {uid:i for i,uid in enumerate(beh.User_ID.unique())}

2025-08-01 17:36:06.473310: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754069766.744051      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754069766.821244      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using cuda
Sampled behaviors: 31,393 rows
Retained news: 39,673 articles


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/285 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

news_x shape: torch.Size([39673, 502])


In [2]:
########################################################################
# 6. 构建 DataLoader
########################################################################
MAX_LEN = 50

class HistDataset(torch.utils.data.Dataset):
    def __init__(self, user_idx, news_idx, labels):
        self.u, self.n, self.y = user_idx, news_idx, labels
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        uid = self.u[i].item()
        raw = user_hist.get(uid, [])
        hids = [hid for hid in raw if 0<=hid<news_x.size(0)] or [self.n[i].item()]
        seq = torch.tensor(hids, dtype=torch.long)
        return news_x[seq], news_x[self.n[i].item()], self.y[i], uid

def collate(batch):
    h, c, y, u = zip(*batch)
    lengths = torch.tensor([x.size(0) for x in h])
    return (
        nn.utils.rnn.pad_sequence(h, batch_first=True),
        torch.stack(c), torch.stack(y), lengths, torch.tensor(u)
    )

def build_loaders(df_train, df_val, neg_ratio=4):
    global user_hist
    user_hist = {}
    for row in df_train.itertuples(index=False):
        uid = uid2idx[row.User_ID]
        user_hist.setdefault(uid, []).extend(
            [news2idx[n] for n in row.History_parsed if n in news2idx]
        )
        user_hist[uid] = user_hist[uid][-MAX_LEN:]
    pu,pn,nu,nn = generate_balanced_samples(df_train, uid2idx, news2idx, neg_ratio)
    tr = (
        torch.tensor(pu+nu), torch.tensor(pn+nn),
        torch.tensor([1]*len(pu)+[0]*len(nu), dtype=torch.float)
    )
    va_u,va_n,va_l = [],[],[]
    for row in df_val.itertuples(index=False):
        uidx = uid2idx[row.User_ID]
        for nid,lbl in row.Impressions_parsed:
            if nid in news2idx:
                va_u.append(uidx); va_n.append(news2idx[nid]); va_l.append(lbl)
    va = (
        torch.tensor(va_u), torch.tensor(va_n),
        torch.tensor(va_l, dtype=torch.float)
    )
    return (
        DataLoader(HistDataset(*tr), batch_size=256, shuffle=True, collate_fn=collate),
        DataLoader(HistDataset(*va), batch_size=2048, shuffle=False, collate_fn=collate)
    )

########################################################################
# 7. 模型定义
########################################################################
class NewsRNN(nn.Module):
    def __init__(self, in_dim, hidden_size=128, cell="GRU"):
        super().__init__()
        Cell = {"RNN":nn.RNN, "GRU":nn.GRU, "LSTM":nn.LSTM}[cell]
        self.rnn = Cell(input_size=in_dim, hidden_size=hidden_size, batch_first=True)
        self.proj = nn.Linear(in_dim, hidden_size)
        self.cell = cell
        self.rnn.flatten_parameters()
    def forward(self, seq, lens, cand):
        seq = seq.contiguous(); self.rnn.flatten_parameters()
        packed = nn.utils.rnn.pack_padded_sequence(seq, lens, batch_first=True, enforce_sorted=False)
        h = self.rnn(packed)[1]
        if self.cell=="LSTM": h = h[0]
        u = h.squeeze(0); v = self.proj(cand)
        return (u*v).sum(-1)

########################################################################
# 8. 评价指标
########################################################################
def calc_metrics(y_true, y_pred, users, k: int=10):
    auc = roc_auc_score(y_true, y_pred)
    df = pd.DataFrame({"u":users, "y":y_true, "p":y_pred})
    hits,prec,rec,mrr,ndcg = [],[],[],[],[]
    def ndcg_k(r,k):
        r = np.asarray(r)[:k]
        dcg = (r/np.log2(np.arange(2,len(r)+2))).sum()
        idcg= (1/np.log2(np.arange(2,int(r.sum())+2))).sum()
        return dcg/idcg if idcg>0 else 0
    for _,g in df.groupby("u"):
        y = g.sort_values("p",ascending=False).y.values
        if y.sum()==0: continue
        hits.append(int(y[:k].sum()>0))
        prec.append(y[:k].mean())
        rec.append(y[:k].sum()/y.sum())
        mrr.append(1/(np.argmax(y==1)+1))
        ndcg.append(ndcg_k(y,k))
    return {"hit_rate":np.mean(hits),"precision":np.mean(prec),"recall":np.mean(rec),"MRR":np.mean(mrr),"NDCG":np.mean(ndcg),"AUC":auc}

########################################################################
# 9. 训练 & 对比
########################################################################
def train_eval(cell, train_loader, val_loader, n_epochs=10):
    model = NewsRNN(news_x.size(1), hidden_size=128, cell=cell).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(n_epochs):
        total = len(train_loader); interval = max(1,total//10)
        print(f"[{cell}] Epoch {epoch+1}/{n_epochs} started")
        model.train(); rloss=0
        for i,(seq,cand,y,lens,_) in enumerate(train_loader,1):
            seq,cand,y=seq.to(DEVICE),cand.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); loss = F.binary_cross_entropy_with_logits(model(seq,lens,cand),y)
            loss.backward(); opt.step(); rloss+=loss.item()
            if i%interval==0 or i==total:
                print(f"  ├─ {100*i/total:.1f}% avg_loss={rloss/i:.4f}")
        print(f"[{cell}] Epoch {epoch+1} done avg_loss={rloss/total:.4f}")
    model.eval(); Y,P,U=[],[],[]
    with torch.no_grad():
        for seq,cand,y,lens,uid in val_loader:
            seq,cand=seq.to(DEVICE),cand.to(DEVICE)
            logits=torch.sigmoid(model(seq,lens,cand)).cpu()
            Y.append(y);P.append(logits);U.append(uid)
    return calc_metrics(torch.cat(Y).numpy(), torch.cat(P).numpy(), torch.cat(U).numpy())

########################################################################
# 10. 主流程：Hold-out 训练 & 评估
########################################################################
train_loader,val_loader = build_loaders(df_train, df_val)
results = {}
for cell in ["GRU","LSTM"]:
    print(f"\n⏳ Training {cell} ...")
    results[cell] = train_eval(cell,train_loader,val_loader)
    print(results[cell])
print("\n===== Summary =====")
for c,m in results.items(): print(f"{c}: {m}")


⏳ Training GRU ...
[GRU] Epoch 1/10 started
  ├─ 10.0% avg_loss=0.5147
  ├─ 20.0% avg_loss=0.4478
  ├─ 30.0% avg_loss=0.4141
  ├─ 40.0% avg_loss=0.3937
  ├─ 50.0% avg_loss=0.3789
  ├─ 60.0% avg_loss=0.3690
  ├─ 69.9% avg_loss=0.3604
  ├─ 79.9% avg_loss=0.3539
  ├─ 89.9% avg_loss=0.3480
  ├─ 99.9% avg_loss=0.3430
  ├─ 100.0% avg_loss=0.3430
[GRU] Epoch 1 done avg_loss=0.3430
[GRU] Epoch 2/10 started
  ├─ 10.0% avg_loss=0.2764
  ├─ 20.0% avg_loss=0.2794
  ├─ 30.0% avg_loss=0.2779
  ├─ 40.0% avg_loss=0.2786
  ├─ 50.0% avg_loss=0.2785
  ├─ 60.0% avg_loss=0.2782
  ├─ 69.9% avg_loss=0.2775
  ├─ 79.9% avg_loss=0.2772
  ├─ 89.9% avg_loss=0.2758
  ├─ 99.9% avg_loss=0.2754
  ├─ 100.0% avg_loss=0.2754
[GRU] Epoch 2 done avg_loss=0.2754
[GRU] Epoch 3/10 started
  ├─ 10.0% avg_loss=0.2426
  ├─ 20.0% avg_loss=0.2445
  ├─ 30.0% avg_loss=0.2449
  ├─ 40.0% avg_loss=0.2468
  ├─ 50.0% avg_loss=0.2477
  ├─ 60.0% avg_loss=0.2490
  ├─ 69.9% avg_loss=0.2495
  ├─ 79.9% avg_loss=0.2504
  ├─ 89.9% avg_loss=0.2